# VideoToSMPL — Notebook 03: End-to-end pipeline

**Video in → G1 PKL + preview MP4 out.**

Colab free T4 · zero install · single person, static camera, ≤ 30 s clip recommended.

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run all**
3. Upload video when prompted

## 1. Setup (torch + pytorch3d + GVHMR + GMR)

In [ ]:
%cd /content
import sys, subprocess
print(f'Python {sys.version.split()[0]}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# ── clone ──────────────────────────────────────────────────────────────
![ -d GVHMR ] || git clone --depth 1 https://github.com/zju3dv/GVHMR.git
![ -d GMR ]   || git clone --depth 1 https://github.com/YanjieZe/GMR.git

# ── torch 2.3.0 + CUDA 12.1 (GVHMR's pin) ──────────────────────────────
!pip install -q torch==2.3.0+cu121 torchvision==0.18.0+cu121 --index-url https://download.pytorch.org/whl/cu121

# ── pytorch3d: prebuilt wheel only exists for py310+cu121+torch230 ─────
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
if py_tag == 'cp310':
    print('→ installing prebuilt pytorch3d wheel')
    !pip install -q https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl
else:
    print(f'→ Colab is {py_tag}; building pytorch3d from source (~10 min, one-time)')
    !pip install -q 'fvcore' 'iopath'
    !pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/pytorch3d.git@V0.7.6'

# ── remaining GVHMR deps (strip the pytorch3d URL line, we handled it) ─
!grep -v 'pytorch3d' GVHMR/requirements.txt > /tmp/gvhmr_reqs.txt
!pip install -q -r /tmp/gvhmr_reqs.txt
!pip install -q -e GVHMR --no-deps

# ── GMR + rendering deps ───────────────────────────────────────────────
!pip install -q -e GMR
!pip install -q 'mujoco>=3.1' 'imageio[ffmpeg]' scipy smplx

import torch
assert torch.cuda.is_available(), 'CUDA not visible — switch runtime to T4 GPU'
print(f'\n✓ torch {torch.__version__} · CUDA {torch.version.cuda} · {torch.cuda.get_device_name(0)}')

## 2. Download GVHMR weights + SMPL-X

SMPL neutral + GVHMR/ViTPose/HMR2/YOLO from public HuggingFace mirror. SMPL-X needs a registration-gated file — the notebook prompts if not already cached.

In [ ]:
import urllib.request
from pathlib import Path
CKPT = Path('/content/GVHMR/inputs/checkpoints')
for sub in ['gvhmr','hmr2','vitpose','yolo','body_models/smpl','body_models/smplx']:
    (CKPT/sub).mkdir(parents=True, exist_ok=True)

WEIGHTS = {
  'gvhmr/gvhmr_siga24_release.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt',
  'hmr2/epoch=10-step=25000.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt',
  'vitpose/vitpose-h-multi-coco.pth':'https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth',
  'yolo/yolov8x.pt':'https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt',
  'body_models/smpl/SMPL_NEUTRAL.pkl':'https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl',
}
for rel, url in WEIGHTS.items():
    dest = CKPT / rel
    if dest.exists() and dest.stat().st_size > 1024:
        print(f'✓ cached {dest.name}')
        continue
    print(f'↓ {dest.name}')
    urllib.request.urlretrieve(url, dest)

# SMPL-X (registration-gated — can't auto-download). Prompt user if missing.
SMPLX = CKPT / 'body_models/smplx/SMPLX_NEUTRAL.npz'
if not SMPLX.exists():
    print('\n⚠ SMPLX_NEUTRAL.npz is required for retargeting but not freely downloadable.')
    print('  Register at https://smpl-x.is.tue.mpg.de/ → download SMPL-X v1.1 Neutral model (.npz).')
    print('  Upload it in the next cell (or skip if you only want the SMPL extraction step).')
else:
    print(f'✓ cached {SMPLX.name}')

print('\nWeights ready.')

### 2b. (optional) Upload SMPL-X if missing

Skip this cell if already cached above.

In [ ]:
from pathlib import Path
SMPLX = Path('/content/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz')
GMR_SMPLX = Path('/content/GMR/assets/body_models/smplx/SMPLX_NEUTRAL.npz')
if not SMPLX.exists():
    from google.colab import files
    print('Upload SMPLX_NEUTRAL.npz (from smpl-x.is.tue.mpg.de)')
    up = files.upload()
    fname = list(up.keys())[0]
    SMPLX.parent.mkdir(parents=True, exist_ok=True)
    GMR_SMPLX.parent.mkdir(parents=True, exist_ok=True)
    SMPLX.write_bytes(up[fname])
    GMR_SMPLX.write_bytes(up[fname])
    print(f'✓ saved {SMPLX.stat().st_size/1e6:.1f} MB')
else:
    # mirror into GMR too if not already there
    if not GMR_SMPLX.exists():
        GMR_SMPLX.parent.mkdir(parents=True, exist_ok=True)
        import shutil; shutil.copy2(SMPLX, GMR_SMPLX)
    print('✓ SMPL-X already cached')

## 3. Upload your video

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
video = Path('/content') / list(uploaded.keys())[0]
print(f'{video.name}  ({video.stat().st_size/1e6:.1f} MB)')

## 4. Extract (GVHMR) + Retarget (GMR)

In [ ]:
import subprocess, time
from pathlib import Path

%cd /content/GVHMR
t0 = time.time()
r = subprocess.run(['python', 'tools/demo/demo.py', '--video', str(video), '-s'],
                   capture_output=True, text=True)
print('\n'.join(r.stdout.splitlines()[-15:]))
if r.returncode != 0:
    print('STDERR:\n' + r.stderr[-2000:])
    raise RuntimeError(f'GVHMR failed ({r.returncode})')
pt = Path('/content/GVHMR/outputs/demo') / video.stem / 'hmr4d_results.pt'
assert pt.exists(), f'missing {pt}'
print(f'\n✓ GVHMR {time.time()-t0:.0f}s → {pt.name}')

%cd /content/GMR
pkl = Path('/content') / (video.stem + '_g1.pkl')
t0 = time.time()
r = subprocess.run(['python', 'scripts/gvhmr_to_robot.py',
                    '--gvhmr_pt', str(pt),
                    '--robot', 'unitree_g1',
                    '--save_path', str(pkl)],
                   capture_output=True, text=True)
print('\n'.join(r.stdout.splitlines()[-15:]))
if r.returncode != 0:
    print('STDERR:\n' + r.stderr[-2000:])
    raise RuntimeError(f'GMR failed ({r.returncode})')
print(f'\n✓ Retarget {time.time()-t0:.0f}s → {pkl.name}')

## 5. Render MuJoCo preview

In [ ]:
import os; os.environ.setdefault('MUJOCO_GL', 'egl')
import pickle, numpy as np, imageio.v2 as imageio, mujoco
from pathlib import Path
from general_motion_retargeting import ROBOT_XML_DICT

with open(pkl, 'rb') as f: d = pickle.load(f)
model = mujoco.MjModel.from_xml_path(str(ROBOT_XML_DICT['unitree_g1']))
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=720, width=1280)
cam = mujoco.MjvCamera(); cam.distance=3.0; cam.azimuth=90; cam.elevation=-15; cam.lookat[:]=[0,0,0.9]

preview = Path('/content') / (video.stem + '_g1.mp4')
w = imageio.get_writer(str(preview), fps=int(d['fps']), codec='libx264', quality=8)
for i in range(len(d['root_pos'])):
    qw,qx,qy,qz = d['root_rot'][i][[3,0,1,2]]
    data.qpos[:3] = d['root_pos'][i]
    data.qpos[3:7] = [qw, qx, qy, qz]
    data.qpos[7:] = d['dof_pos'][i]
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=cam)
    w.append_data(np.asarray(renderer.render()))
w.close(); renderer.close()
print(f'✓ preview → {preview.name}  ({preview.stat().st_size/1e6:.1f} MB)')

## 6. Download artifacts

In [ ]:
from google.colab import files
files.download(str(pt))
files.download(str(pkl))
files.download(str(preview))